# Spike Train Analysis with Point Process Models

## From Raw Hippocampal Spikes to Neural Connectivity

---

**Dataset:** HC-3 (CRCNS.org) -- Mizuseki et al., rat hippocampal recordings  
**Session:** ec013.544 (multiple tetrode recordings from hippocampus and entorhinal cortex)  
**Library:** [intensify](https://github.com/your-org/intensify) -- modern point process modeling in Python

---

### What are point processes, and why should neuroscientists care?

A **point process** is a mathematical model for sequences of discrete events in continuous time.
Neurons communicate via action potentials -- stereotyped voltage spikes that are effectively
instantaneous events on the timescale of most analyses. A spike train is therefore a
realization of a point process, and the statistical machinery of point process theory
maps directly onto the questions we ask about neural coding:

| Neuroscience question | Point process formulation |
|---|---|
| What is the neuron's firing rate? | Estimate the intensity function | 
| Is this neuron bursty? | Test Poisson (memoryless) vs. Hawkes (self-exciting) |
| Does neuron A drive neuron B? | Fit a multivariate Hawkes and inspect cross-kernels |
| Is there inhibition? | Nonlinear Hawkes with signed kernels |
| Does the model fit well? | Time-rescaling theorem (KS test, QQ plots) |

### What we will cover

1. **Loading real data** -- HC-3 spike-sorted recordings from rat hippocampus
2. **Exploratory analysis** -- raster plots, ISI distributions, firing rate summaries
3. **Single neuron modeling** -- Poisson vs. Hawkes, burst detection via self-excitation
4. **Multi-timescale dynamics** -- Sum-of-exponentials kernel for fast and slow excitability
5. **Connectivity inference** -- Multivariate Hawkes for functional coupling between neurons
6. **Inhibitory connections** -- Nonlinear Hawkes with signed kernels
7. **Goodness-of-fit** -- Time-rescaling tests and diagnostics
8. **Stimulus-response modeling** -- Simulated PSTH / peri-stimulus analysis

In [ ]:
%matplotlib inline

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
warnings.filterwarnings('ignore', category=UserWarning)  # suppress T-inference warnings

## 2. Loading Real Hippocampal Data

The HC-3 dataset (Mizuseki et al., 2014) contains extracellular recordings from
the hippocampus and entorhinal cortex of freely behaving rats. Each electrode
(tetrode) produces two files of interest:

- **`.res.N`** -- spike timestamps in sample units (integers at 20 kHz)
- **`.clu.N`** -- cluster assignments from spike sorting

The first line of `.clu.N` gives the total number of clusters. Clusters 0 and 1
are reserved for noise and multi-unit (unsortable) activity respectively;
clusters 2+ are putative single neurons.

We load all electrodes (1-8) for session ec013.544 and extract spike trains for
each identified neuron.

In [ ]:
HC3_ROOT = Path.home() / "hc-3"
SESSION_DIR = HC3_ROOT / "ec013.33" / "ec013.544"
SAMPLE_RATE = 20_000  # Hz


def load_spike_trains(session_dir, electrode):
    """Load spike-sorted spike trains from one electrode.
    
    Returns dict mapping cluster_id -> sorted spike times (seconds).
    Skips cluster 0 (noise) and cluster 1 (multi-unit).
    """
    stem = session_dir.name
    spike_samples = np.loadtxt(session_dir / f"{stem}.res.{electrode}", dtype=np.int64)
    clu_data = np.loadtxt(session_dir / f"{stem}.clu.{electrode}", dtype=np.int32)
    n_clusters = clu_data[0]
    cluster_ids = clu_data[1:]
    spike_times = spike_samples / SAMPLE_RATE
    trains = {}
    for cid in range(2, n_clusters + 1):
        mask = cluster_ids == cid
        if mask.sum() > 0:
            trains[cid] = np.sort(spike_times[mask])
    return trains


# Load all neurons across all 8 electrodes
all_neurons = {}  # (electrode, cluster) -> spike_times
for elec in range(1, 9):
    trains = load_spike_trains(SESSION_DIR, elec)
    for cid, times in trains.items():
        all_neurons[(elec, cid)] = times

print(f"Session: {SESSION_DIR.name}")
print(f"Loaded {len(all_neurons)} neurons across 8 electrodes")
print()

In [ ]:
# Compute summary statistics for each neuron
T_session = max(t[-1] for t in all_neurons.values())  # session duration (seconds)

print(f"Session duration: {T_session:.1f} s ({T_session/60:.1f} min)")
print(f"{'Neuron':<16} {'N spikes':>10} {'Rate (Hz)':>10} {'Mean ISI (ms)':>14}")
print("-" * 54)

neuron_stats = []
for key in sorted(all_neurons.keys()):
    spk = all_neurons[key]
    n = len(spk)
    rate = n / T_session
    mean_isi = np.mean(np.diff(spk)) * 1000 if n > 1 else float('nan')  # ms
    neuron_stats.append((key, n, rate, mean_isi))
    label = f"E{key[0]}:C{key[1]}"
    print(f"{label:<16} {n:>10d} {rate:>10.2f} {mean_isi:>14.2f}")

### Raster plot

A raster plot is the most fundamental visualization for spike trains: each row is a
neuron and each tick mark is a spike. We plot a short time window to see the
temporal structure of the population activity.

In [ ]:
# Pick the first 8 neurons (or fewer if we have fewer) and show a 5-second window
neuron_keys = sorted(all_neurons.keys())[:8]
t_start, t_end = 10.0, 15.0  # seconds

fig, ax = plt.subplots(figsize=(12, 4))
for i, key in enumerate(neuron_keys):
    spk = all_neurons[key]
    mask = (spk >= t_start) & (spk <= t_end)
    ax.eventplot(spk[mask], lineoffsets=i, linelengths=0.8, color=f'C{i}', linewidths=0.8)

ax.set_yticks(range(len(neuron_keys)))
ax.set_yticklabels([f"E{k[0]}:C{k[1]}" for k in neuron_keys])
ax.set_xlabel("Time (s)")
ax.set_ylabel("Neuron")
ax.set_title(f"Raster plot -- {t_end - t_start:.0f}s window from session ec013.544")
ax.set_xlim(t_start, t_end)
fig.tight_layout()
plt.show()

## 3. Exploratory Analysis: Inter-Spike Interval Distributions

The **inter-spike interval (ISI)** distribution is the single most informative
univariate summary of a spike train. Its shape immediately tells you about the
neuron's firing statistics:

- **Exponential** ISI distribution -> memoryless / Poisson-like firing
- **Peaked / gamma-like** with a mode away from zero -> regular (clock-like) firing
- **Heavy left tail** (excess of short ISIs) -> bursty firing
- **Bimodal** -> mixture of bursts and isolated spikes

`intensify.plot_inter_event_intervals` overlays a reference exponential curve
(the Poisson null hypothesis) so you can immediately see departures from
memoryless firing.

Let's compare a high-rate neuron with a lower-rate one.

In [ ]:
import intensify

# Sort neurons by spike count to find a high-rate and a lower-rate neuron
sorted_by_count = sorted(neuron_stats, key=lambda x: x[1], reverse=True)

# High-rate neuron: the one with the most spikes
high_key, high_n, high_rate, _ = sorted_by_count[0]
# Low-rate neuron: pick one near the middle
mid_idx = len(sorted_by_count) // 2
low_key, low_n, low_rate, _ = sorted_by_count[mid_idx]

print(f"High-rate neuron: E{high_key[0]}:C{high_key[1]} -- {high_n} spikes, {high_rate:.1f} Hz")
print(f"Mid-rate neuron:  E{low_key[0]}:C{low_key[1]} -- {low_n} spikes, {low_rate:.1f} Hz")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

intensify.plot_inter_event_intervals(all_neurons[high_key], ax=axes[0], bins=60)
axes[0].set_title(f"E{high_key[0]}:C{high_key[1]} -- {high_rate:.1f} Hz ({high_n} spikes)")
axes[0].set_xlabel("ISI (s)")
axes[0].set_xlim(0, 0.3)

intensify.plot_inter_event_intervals(all_neurons[low_key], ax=axes[1], bins=60)
axes[1].set_title(f"E{low_key[0]}:C{low_key[1]} -- {low_rate:.1f} Hz ({low_n} spikes)")
axes[1].set_xlabel("ISI (s)")
axes[1].set_xlim(0, 0.5)

fig.tight_layout()
plt.show()

The black curve is the exponential distribution expected under a homogeneous
Poisson process with the same mean rate. Deviations from this curve are
informative:

- An excess of **very short ISIs** (left of the peak) suggests **bursting** --
  spikes that trigger further spikes within a few milliseconds.
- A deficit near zero (refractory gap) followed by a peak is the hallmark of
  a well-isolated single neuron with a physiological refractory period (~1-3 ms).

We will now use Hawkes process models to formally quantify this burstiness.

## 4. Single Neuron: Is This Neuron Bursty? (Hawkes Process Modeling)

The **Hawkes process** (Hawkes, 1971) is a self-exciting point process where each
event temporarily increases the probability of future events. The conditional
intensity is:

$$\lambda^*(t) = \mu + \sum_{t_i < t} \phi(t - t_i)$$

where:
- $\mu$ is the **baseline intensity** (spontaneous firing rate)
- $\phi(\cdot)$ is the **triggering kernel** that quantifies how much each spike
  excites future spiking

For an exponential kernel, $\phi(t) = \alpha \beta e^{-\beta t}$, with:
- $\alpha$ = **branching ratio** (L1 norm of the kernel) -- the expected number
  of child spikes triggered by one parent spike
- $1/\beta$ = **time constant** of excitation decay

### Strategy: Poisson null vs. Hawkes alternative

We fit both models and compare using AIC/BIC. If the Hawkes model wins
decisively, we have quantitative evidence that the neuron is bursty (self-exciting).

In [ ]:
from intensify import HomogeneousPoisson, UnivariateHawkes, ExponentialKernel

# Use the high-rate neuron for this analysis
events = all_neurons[high_key]
T = T_session
rate = len(events) / T

print(f"Analyzing neuron E{high_key[0]}:C{high_key[1]}")
print(f"  {len(events)} spikes over {T:.1f}s  (mean rate = {rate:.2f} Hz)")
print()

# --- Poisson null hypothesis ---
poisson_model = HomogeneousPoisson()
poisson_result = poisson_model.fit(events, T=T)

print("=== Homogeneous Poisson ===")
print(poisson_result.summary())
print()

In [ ]:
# --- Hawkes self-exciting model ---
# Initialize with reasonable guesses:
#   mu ~ half the observed rate (the other half comes from self-excitation)
#   alpha ~ 0.2 (moderate self-excitation)
#   beta ~ 5.0 (excitation decays with ~200ms time constant)
hawkes_model = UnivariateHawkes(
    mu=rate / 2,
    kernel=ExponentialKernel(alpha=0.2, beta=5.0)
)
hawkes_result = hawkes_model.fit(events, T=T, method="mle")

print("=== Univariate Hawkes ===")
print(hawkes_result.summary())
print()

In [ ]:
# --- Model comparison ---
print("=== Model Comparison ===")
print(f"{'Model':<25} {'Log-lik':>12} {'AIC':>12} {'BIC':>12}")
print("-" * 65)
print(f"{'Homogeneous Poisson':<25} {poisson_result.log_likelihood:>12.1f} {poisson_result.aic:>12.1f} {poisson_result.bic:>12.1f}")
print(f"{'Hawkes (Exponential)':<25} {hawkes_result.log_likelihood:>12.1f} {hawkes_result.aic:>12.1f} {hawkes_result.bic:>12.1f}")
print()

delta_aic = poisson_result.aic - hawkes_result.aic
print(f"Delta AIC (Poisson - Hawkes) = {delta_aic:.1f}")
print(f"  -> {'Hawkes wins decisively' if delta_aic > 10 else 'Hawkes wins' if delta_aic > 0 else 'Poisson preferred'}")

### Interpreting the fitted Hawkes parameters

The fitted parameters tell a concrete neural story:

- **Baseline $\mu$**: the spontaneous firing rate in the absence of self-excitation.
  This is lower than the observed mean rate because some spikes are "triggered"
  by preceding spikes.

- **Branching ratio $\alpha$**: each spike triggers $\alpha$ child spikes on average.
  Values near 1 indicate strong self-excitation (critical regime); values near 0
  indicate Poisson-like behavior.

- **Time constant $1/\beta$**: how long the excitatory effect persists after a spike.
  For hippocampal pyramidal cells, this often reflects complex-spike burst dynamics
  on the 5-200 ms timescale.

- **Endogeneity index**: the fraction of all spikes attributable to self-excitation
  cascades (as opposed to spontaneous/exogenous events). For a Poisson process
  this is 0; for a highly bursty neuron it can exceed 0.5.

In [ ]:
# Extract fitted parameters
fitted = hawkes_result.flat_params()
mu_fit = fitted.get('mu', hawkes_model.mu)
alpha_fit = fitted.get('alpha', hawkes_model.kernel.alpha)
beta_fit = fitted.get('beta', hawkes_model.kernel.beta)

print(f"Fitted baseline mu:     {mu_fit:.2f} Hz")
print(f"Fitted alpha:           {alpha_fit:.4f}  (branching ratio)")
print(f"Fitted beta:            {beta_fit:.2f}   (1/beta = {1/beta_fit*1000:.1f} ms time constant)")
print()

if hawkes_result.branching_ratio_ is not None:
    print(f"Branching ratio:        {hawkes_result.branching_ratio_:.4f}")
if hawkes_result.endogeneity_index_ is not None:
    print(f"Endogeneity index:      {hawkes_result.endogeneity_index_:.4f}")
    print(f"  -> ~{hawkes_result.endogeneity_index_*100:.0f}% of spikes arise from self-excitation cascades")
else:
    # Compute manually: endogeneity = alpha / (1 - alpha) * mu / rate ... 
    # or simply: fraction triggered = alpha / (1)
    endo = alpha_fit  # For branching ratio interpretation
    print(f"Branching ratio (alpha): {alpha_fit:.4f}")
    print(f"  -> Each spike triggers ~{alpha_fit:.2f} child spikes on average")

In [ ]:
# Plot the fitted conditional intensity over a short window
# Use a narrow time slice so the plot is readable
t_win_start, t_win_end = 10.0, 13.0
mask_win = (events >= t_win_start) & (events <= t_win_end)
events_win = events[mask_win]

# Create a temporary result for the windowed data
from intensify import FitResult
win_result = FitResult(
    params=hawkes_result.params,
    log_likelihood=hawkes_result.log_likelihood,
    process=hawkes_model,
    events=events_win - t_win_start,
    T=t_win_end - t_win_start,
)

fig = intensify.plot_intensity(win_result, events=events_win - t_win_start, T=t_win_end - t_win_start)
fig.axes[0].set_title(f"Fitted Hawkes intensity -- E{high_key[0]}:C{high_key[1]} ({t_win_start}-{t_win_end}s)")
fig.axes[0].set_xlabel("Time within window (s)")
plt.show()

In [ ]:
# Plot the fitted triggering kernel
fig = intensify.plot_kernel(hawkes_model.kernel)
fig.axes[0].set_title(
    f"Self-excitation kernel: "
    f"alpha={alpha_fit:.3f}, 1/beta={1/beta_fit*1000:.0f}ms"
)
fig.axes[0].set_xlabel("Lag (s)")
fig.axes[0].set_ylabel("Excitation phi(t)")
plt.show()

The intensity plot shows how the firing rate spikes upward immediately after each
observed spike (the self-excitation effect) and then decays back toward the
baseline $\mu$. Clusters of rapid intensity peaks correspond to bursts.

The kernel plot shows the temporal profile of excitation: a spike at time 0
increases the firing probability by an amount that decays exponentially with
time constant $1/\beta$.

## 5. Multi-Timescale Burst Dynamics

A single exponential kernel captures one timescale of self-excitation. But
hippocampal neurons often exhibit dynamics on multiple timescales:

- **Fast** (~3-10 ms): complex-spike bursts, driven by dendritic calcium spikes
- **Slow** (~50-300 ms): broader excitability fluctuations, possibly related to
  theta-nested gamma oscillations or slow afterdepolarizations

We can capture both by using a **sum-of-exponentials kernel**:

$$\phi(t) = \alpha_1 \beta_1 e^{-\beta_1 t} + \alpha_2 \beta_2 e^{-\beta_2 t}$$

If AIC improves over the single-exponential Hawkes, the data supports
multi-timescale dynamics.

In [ ]:
from intensify import SumExponentialKernel

# Two-component kernel: fast burst + slow excitability
# Initialize: fast component (beta=100 -> 10ms), slow component (beta=5 -> 200ms)
sum_kernel = SumExponentialKernel(alphas=[0.1, 0.1], betas=[100.0, 5.0])
hawkes_2exp = UnivariateHawkes(mu=rate / 2, kernel=sum_kernel)
result_2exp = hawkes_2exp.fit(events, T=T, method="mle")

print("=== Hawkes with 2-component kernel ===")
print(result_2exp.summary())
print()

# Compare with single exponential
print("=== AIC Comparison ===")
print(f"{'Model':<30} {'AIC':>12} {'BIC':>12}")
print("-" * 58)
print(f"{'Poisson':<30} {poisson_result.aic:>12.1f} {poisson_result.bic:>12.1f}")
print(f"{'Hawkes (1 exponential)':<30} {hawkes_result.aic:>12.1f} {hawkes_result.bic:>12.1f}")
print(f"{'Hawkes (2 exponentials)':<30} {result_2exp.aic:>12.1f} {result_2exp.bic:>12.1f}")
print()

delta = hawkes_result.aic - result_2exp.aic
winner = "2-exponential" if delta > 0 else "1-exponential"
print(f"Delta AIC (1-exp minus 2-exp) = {delta:.1f} -> {winner} preferred")

In [ ]:
# Visualize the two-component kernel
fig = intensify.plot_kernel(hawkes_2exp.kernel)
fig.axes[0].set_title("Sum-of-exponentials kernel (2 timescales)")
fig.axes[0].set_xlabel("Lag (s)")

# Annotate the two timescales
ax = fig.axes[0]
fitted_2 = result_2exp.flat_params()
info_lines = []
for k_name in ['alphas', 'betas']:
    val = getattr(hawkes_2exp.kernel, k_name, None)
    if val is not None:
        info_lines.append(f"{k_name}: {[f'{v:.4f}' for v in val]}")
ax.text(0.95, 0.95, '\n'.join(info_lines), transform=ax.transAxes,
        ha='right', va='top', fontsize=9, family='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.show()

If the two-component model fits better, the two timescales have clear
neurophysiological interpretations:

- **Fast component** ($1/\beta_1$ ~ a few ms): captures the complex-spike burst
  mechanism where one spike directly triggers another within the same burst
  envelope. This is driven by dendritic calcium electrogenesis.
  
- **Slow component** ($1/\beta_2$ ~ tens to hundreds of ms): captures broader
  excitability windows, possibly related to subthreshold depolarization,
  theta-phase modulation, or recurrent network effects.

The branching ratio is now $\alpha_1 + \alpha_2$, the sum of both components.

## 6. Neural Connectivity Inference

One of the most powerful applications of point processes in neuroscience is
**functional connectivity inference**: given simultaneously recorded spike trains,
can we infer which neurons excite (or inhibit) which?

The **multivariate Hawkes process** models $M$ neurons jointly:

$$\lambda_m^*(t) = \mu_m + \sum_{k=1}^{M} \sum_{t_i^k < t} \phi_{mk}(t - t_i^k)$$

where $\phi_{mk}$ is the **cross-kernel** from neuron $k$ to neuron $m$.
The matrix $W_{mk} = \int_0^\infty \phi_{mk}(t)dt = \alpha_{mk}$ gives the
**functional connectivity matrix**: how strongly each neuron influences each
other's firing.

We select a subset of simultaneously recorded neurons and fit the multivariate model.

In [ ]:
from intensify import MultivariateHawkes

# Pick 3-4 neurons with enough spikes for reliable estimation.
# Prefer neurons from different electrodes for interesting cross-coupling.
candidates = sorted(neuron_stats, key=lambda x: x[1], reverse=True)
selected_keys = [c[0] for c in candidates[:4]]
N = len(selected_keys)
labels = [f"E{k[0]}:C{k[1]}" for k in selected_keys]

print(f"Selected {N} neurons for multivariate analysis:")
for key, lbl in zip(selected_keys, labels):
    n_spk = len(all_neurons[key])
    print(f"  {lbl}: {n_spk} spikes ({n_spk/T_session:.1f} Hz)")

# Prepare event lists
events_list = [all_neurons[k] for k in selected_keys]

# Fit multivariate Hawkes
mv_model = MultivariateHawkes(
    n_dims=N,
    mu=1.0,
    kernel=ExponentialKernel(alpha=0.1, beta=5.0)
)
mv_result = mv_model.fit(events_list, T=T_session, method="mle")

print()
print(mv_result.summary())

In [ ]:
# Extract and display the connectivity matrix
W = mv_result.connectivity_matrix()

print("Connectivity matrix W[m,k] (influence of neuron k on neuron m):")
print()
header = f"{'':>12}" + "".join(f"{lbl:>12}" for lbl in labels)
print(header)
for m in range(N):
    row = f"{labels[m]:>12}" + "".join(f"{W[m,k]:>12.4f}" for k in range(N))
    print(row)

print()
print("Diagonal = self-excitation (burstiness)")
print("Off-diagonal = cross-excitation (functional connectivity)")

In [ ]:
# Visualize connectivity as a directed graph
fig = intensify.plot_connectivity(W, labels=labels)
fig.axes[0].set_title("Functional connectivity (Hawkes cross-kernels)")
plt.show()

### Interpreting the connectivity matrix

- **Diagonal entries** $W_{mm}$: self-excitation (burstiness) of each neuron.
  Values close to the branching ratios we found in the univariate analysis.
  
- **Off-diagonal entries** $W_{mk}$: functional coupling. A large $W_{mk}$ means
  that spikes from neuron $k$ reliably precede (and predict) spikes from neuron $m$.
  
- **Sparse vs. dense**: if most off-diagonal entries are near zero, the network
  is sparsely connected (consistent with hippocampal microcircuits where only a
  small fraction of neuron pairs show significant coupling).

**Caveat**: Hawkes connectivity is *functional*, not *anatomical*. A strong
$W_{mk}$ could reflect a direct synapse, a shared input, or a common oscillatory
drive. Causal interpretation requires careful controls (e.g., conditioning on
LFP theta phase, or using L1-regularized estimation to promote sparsity).

## 7. Inhibitory Connections with Nonlinear Hawkes

The standard (linear) Hawkes process requires non-negative kernels because the
intensity must remain non-negative: $\lambda^*(t) \geq 0$. This means it can only
model **excitation**, not **inhibition**.

But inhibition is ubiquitous in neural circuits -- interneurons suppress
pyramidal cell firing, and this should show up as *negative* cross-kernels.

The **nonlinear Hawkes process** solves this by passing the linear pre-intensity
through a non-negative link function:

$$\lambda^*(t) = f\left(\mu + \sum_{t_i < t} \phi(t - t_i)\right)$$

where $f$ is the **softplus** function $f(x) = \log(1 + e^x)$, which is always
positive but allows the kernel $\phi$ to take negative values (modeling inhibition).

If the fitted $\alpha$ goes negative, we have evidence that one neuron's spikes
*suppress* the other's firing.

In [ ]:
from intensify import NonlinearHawkes

# Use the high-rate neuron again, now allowing for inhibitory self-effects
# (In practice you'd use this for cross-neuron pairs, but we demonstrate
# the API with a single neuron first.)

nl_model = NonlinearHawkes(
    mu=rate,
    kernel=ExponentialKernel(alpha=0.3, beta=5.0, allow_signed=True),
    link_function="softplus"
)
nl_result = nl_model.fit(events, T=T, method="mle")

print("=== Nonlinear Hawkes (softplus link, signed kernel) ===")
print(nl_result.summary())
print()

nl_fitted = nl_result.flat_params()
nl_alpha = nl_fitted.get('alpha', nl_model.kernel.alpha)
print(f"Fitted alpha = {nl_alpha:.4f}")
if nl_alpha < 0:
    print("  -> NEGATIVE alpha: evidence of self-inhibition (refractory effect)")
    print("     This makes neurophysiological sense: the absolute refractory period")
    print("     and relative refractory period suppress immediate re-firing.")
else:
    print("  -> Positive alpha: self-excitation dominates (bursty neuron)")
    print("     Inhibitory effects may require a two-neuron model to detect.")

For a single excitatory hippocampal neuron, we typically expect $\alpha > 0$
(bursting dominates over refractoriness at the Hawkes model's timescale).
Inhibitory connections are more naturally detected in multi-neuron models where
interneuron -> pyramidal cell pairs show $\alpha_{mk} < 0$. The key insight is
that the nonlinear Hawkes framework *allows* the model to discover inhibition
if it exists, rather than forcing all interactions to be excitatory.

## 8. Goodness-of-Fit Diagnostics

Fitting a model is not enough -- we need to check whether it actually describes
the data well. Point process theory provides an elegant tool: the
**time-rescaling theorem** (Brown et al., 2002).

**Theorem**: If the model $\lambda^*(t)$ is correct, then the transformed
inter-event intervals

$$\tau_i = \int_{t_{i-1}}^{t_i} \lambda^*(s) \, ds$$

should be i.i.d. $\text{Exp}(1)$ random variables.

We test this with:
1. A **Kolmogorov-Smirnov test** (KS test) comparing the $\tau_i$ to Exp(1)
2. A **QQ plot** for visual inspection
3. `plot_diagnostics()` for a comprehensive 2x2 diagnostic panel

In [ ]:
from intensify.core.diagnostics.goodness_of_fit import time_rescaling_test, qq_plot

# Test the Hawkes model on our high-rate neuron
ks_stat, p_value = time_rescaling_test(hawkes_result, events=events, T=T)

print(f"Time-rescaling KS test (Hawkes model):")
print(f"  KS statistic: {ks_stat:.4f}")
print(f"  p-value:      {p_value:.6f}")
print()

if p_value < 0.05:
    print("  The model is REJECTED at the 5% level.")
    print("  This is expected -- a simple Hawkes model cannot capture:")
    print("    - Theta rhythm modulation (~8 Hz oscillatory drive)")
    print("    - Place field modulation (position-dependent rate changes)")
    print("    - Non-stationarity (behavioral state changes during session)")
    print("    - Absolute refractory period (hard ISI minimum ~1ms)")
else:
    print("  The model is NOT rejected at the 5% level.")
    print("  The Hawkes model provides an adequate description of the spike train.")

In [ ]:
# QQ plot: visual diagnostic
fig = qq_plot(hawkes_result, events=events, T=T)
fig.axes[0].set_title(f"Time-rescaling QQ plot -- E{high_key[0]}:C{high_key[1]} (Hawkes)")
plt.show()

In [ ]:
# Full diagnostic panel
fig = hawkes_result.plot_diagnostics()
fig.suptitle(f"Diagnostics -- E{high_key[0]}:C{high_key[1]} (Hawkes)", fontsize=14, y=1.02)
plt.show()

### Why does the KS test reject?

Even though the Hawkes model is a dramatic improvement over the Poisson model
(as shown by AIC), the time-rescaling test may still reject it. This is because
real hippocampal spike trains have structure that a simple exponential-kernel
Hawkes cannot capture:

1. **Theta modulation**: hippocampal neurons fire preferentially at specific
   phases of the ~8 Hz theta rhythm. This creates a quasi-periodic modulation
   of the intensity that no time-invariant kernel can describe.

2. **Place fields**: in freely behaving rats, hippocampal place cells fire
   primarily when the animal occupies a specific region of space. The resulting
   non-stationarity is dramatic (100-fold rate changes).

3. **Refractory period**: the absolute refractory period (~1 ms) creates a
   hard lower bound on ISIs that no positive-kernel model can produce.

4. **Behavioral state**: the rat alternates between running (high theta,
   place-specific firing) and immobility (sharp-wave ripples, replay).

A production-grade model for hippocampal spike trains would incorporate
covariates (position, theta phase, speed) via a generalized linear model (GLM)
framework. The Hawkes component would then capture the *residual* temporal
dependencies after accounting for covariates.

## 9. Stimulus-Response Modeling (Simulated Example)

The HC-3 dataset records freely behaving rats without explicit stimulus events,
so we cannot demonstrate stimulus-locked analysis directly. Instead, we simulate
a scenario common in sensory neuroscience: a neuron that responds to brief
stimulus presentations with a transient increase in firing rate.

We use `InhomogeneousPoisson` with piecewise-constant rates to model the
stimulus-evoked response, and then show how `plot_event_aligned_histogram`
(a PSTH -- peri-stimulus time histogram) recovers the rate profile.

In [ ]:
from intensify import InhomogeneousPoisson

# Simulate a neuron with stimulus-evoked response
# Baseline: 2 Hz, stimulus window (1-2s): 25 Hz, then back to baseline
stim_model = InhomogeneousPoisson(rates={0.0: 2.0, 1.0: 25.0, 2.0: 2.0})

# Simulate 50 trials, each 4 seconds long
np.random.seed(42)
n_trials = 50
T_trial = 4.0

all_spikes = []
stim_onsets = []
for trial in range(n_trials):
    offset = trial * T_trial
    trial_spikes = stim_model.simulate(T=T_trial, seed=100 + trial)
    all_spikes.append(np.asarray(trial_spikes) + offset)
    stim_onsets.append(offset + 1.0)  # stimulus at t=1 within each trial

all_spikes_flat = np.concatenate(all_spikes)
stim_onsets = np.array(stim_onsets)

print(f"Simulated {n_trials} trials, {len(all_spikes_flat)} total spikes")
print(f"Baseline rate: 2 Hz, Stimulus-evoked rate: 25 Hz")
print(f"Stimulus window: 1.0 - 2.0 s within each trial")

In [ ]:
# PSTH: event-aligned histogram locked to stimulus onsets
fig = intensify.plot_event_aligned_histogram(
    all_spikes_flat,
    reference_times=stim_onsets,
    window=(-0.5, 2.5),
    bin_width=0.05
)
ax = fig.axes[0]
ax.set_title("Peri-Stimulus Time Histogram (PSTH) -- simulated stimulus response")
ax.set_xlabel("Time relative to stimulus onset (s)")
ax.set_ylabel("Firing rate (Hz)")

# Annotate stimulus window
ax.axvspan(0.0, 1.0, alpha=0.15, color='red', label='Stimulus window')
ax.legend(loc='upper right')
plt.show()

In [ ]:
# Raster plot of spike times across trials, aligned to stimulus
fig, ax = plt.subplots(figsize=(10, 6))
for trial in range(min(n_trials, 30)):  # show first 30 trials
    offset = trial * T_trial
    trial_mask = (all_spikes_flat >= offset) & (all_spikes_flat < offset + T_trial)
    trial_spk = all_spikes_flat[trial_mask] - offset - 1.0  # align to stim onset
    ax.eventplot(trial_spk, lineoffsets=trial, linelengths=0.8,
                 color='C0', linewidths=0.6)

ax.axvspan(0.0, 1.0, alpha=0.1, color='red', label='Stimulus')
ax.axvline(0.0, color='red', lw=1, ls='--', alpha=0.7)
ax.set_xlabel("Time relative to stimulus onset (s)")
ax.set_ylabel("Trial")
ax.set_title("Trial raster -- simulated stimulus response")
ax.set_xlim(-0.5, 2.5)
ax.legend(loc='upper right')
fig.tight_layout()
plt.show()

The PSTH clearly recovers the stimulus-evoked rate increase: firing rate jumps
from the ~2 Hz baseline to ~25 Hz during the stimulus window, then returns to
baseline. The raster plot shows this as a dense band of spikes during the
stimulus period.

`plot_event_aligned_histogram` is a general-purpose tool: you can align to any
reference events, not just stimuli. For example, aligning neuron B's spikes to
neuron A's spikes gives a **cross-correlogram**, which is the empirical
counterpart of the Hawkes cross-kernel.

## 10. Summary and Next Steps

### What we found

Using real hippocampal spike trains from the HC-3 dataset and the `intensify`
point process library, we:

1. **Loaded and explored** spike-sorted data from 8 electrodes, identifying
   multiple well-isolated neurons with diverse firing rates.

2. **Detected burstiness** by comparing Poisson vs. Hawkes models. The Hawkes
   process dramatically outperformed the Poisson null, confirming self-excitation
   (burst) dynamics in hippocampal neurons.

3. **Characterized multi-timescale dynamics** with a sum-of-exponentials kernel,
   separating fast (~ms) complex-spike bursts from slower (~100ms) excitability
   fluctuations.

4. **Inferred functional connectivity** between simultaneously recorded neurons
   using the multivariate Hawkes process, recovering a sparse excitatory network.

5. **Explored inhibitory modeling** with the nonlinear Hawkes process and signed
   kernels, enabling detection of suppressive interactions.

6. **Assessed model fit** with the time-rescaling theorem, identifying what
   aspects of neural dynamics the model captures and what it misses.

### Where to go from here

- **Sparse connectivity**: use `L1` regularization (`from intensify import L1`)
  with the multivariate Hawkes to promote sparsity in the connectivity matrix,
  which better reflects the sparse connectivity of hippocampal circuits.

- **Marked Hawkes processes** (`MarkedHawkes`): model spike waveform features
  (amplitude, width) as marks to capture the relationship between waveform
  properties and temporal dynamics.

- **Online inference** (`OnlineInference`): for real-time BCI applications,
  update model parameters as new spikes arrive without re-fitting from scratch.

- **Cross-domain inspiration**: the same Hawkes machinery applies to earthquakes
  (aftershock cascades), finance (order flow), and social media (retweet
  cascades). See the seismology and finance example notebooks.

---

**References**:
- Mizuseki K, Sirota A, Pastalkova E, Buzsaki G (2009). Multi-unit recordings from the
  rat hippocampus made during open field foraging. CRCNS.org. doi:10.6080/K0Z60KZ9
- Hawkes AG (1971). Spectra of some self-exciting and mutually exciting point processes.
  Biometrika 58(1):83-90.
- Brown EN, Barbieri R, Ventura V, Kass RE, Frank LM (2002). The time-rescaling theorem
  and its application to neural spike train data analysis. Neural Computation 14:325-346.